In [1]:
import pandas as pd
import numpy as np
# Load dataset
df = pd.read_csv(r"C:\\college_recommendation_system\\CleanData\\Cleaned_tgeapcet_2017_2024.csv")

df["category_gender"] = (
    df["category"].astype(str) + "_" + df["gender"].astype(str)
)

print(df.columns)
df.isnull().sum()
# Drop college_type and year_of_estab columns
df = df.drop(columns=["college_type", "year_of_estab"])
df.isnull().sum()
# Remove extra spaces
df["co_education"] = df["co_education"].str.strip()
df["category"] = df["category"].str.strip()

# Convert to uppercase
df["co_education"] = df["co_education"].str.upper()
df["category"] = df["category"].str.upper()

Index(['inst_code', 'institute_name', 'branch_code', 'branch_name',
       'co_education', 'college_type', 'year_of_estab', 'place', 'dist_code',
       'year', 'cutoff', 'category', 'gender', 'category_gender'],
      dtype='object')


In [2]:
from sklearn.preprocessing import LabelEncoder

le_college = LabelEncoder()
df["inst_code"] = le_college.fit_transform(df["inst_code"])

In [3]:
le_branch = LabelEncoder()
df["branch_code"] = le_branch.fit_transform(df["branch_code"])

In [4]:
df["category"] = df["category"].fillna("EWS")

category_map = {
    "OC":0,
    "BC_A":1,
    "BC_B":2,
    "BC_C":3,
    "BC_D":4,
    "BC_E":5,
    "SC":6,
    "ST":7,
    "EWS":8
}

df["category"] = df["category"].map(category_map)

In [5]:
gender_map = {
    "COED":0,
    "GIRLS":1
}
df["co_education"] = df["co_education"].map(gender_map)

In [6]:
le_location = LabelEncoder()
df["dist_code"] = le_location.fit_transform(df["dist_code"])

In [7]:
gender_map = {
    "Boys": 0,
    "Girls": 1
}

df["gender"] = df["gender"].map(gender_map)

In [8]:
df = df.drop(columns=[
    "institute_name",
    "branch_name",
    "category",
    "gender",
    "place"
])

In [9]:
df["category_gender"] = df["category_gender"].str.strip().str.upper()

In [10]:
from sklearn.preprocessing import LabelEncoder

le_catgen = LabelEncoder()
df["catgen_id"] = le_catgen.fit_transform(df["category_gender"])

In [11]:
df = df.drop(columns=["category_gender"])

In [12]:
df.head()

,inst_code,branch_code,co_education,dist_code,year,cutoff,catgen_id
0,0,20,0,2,2017,26907.0,12
1,0,30,0,2,2017,33871.0,12
2,0,33,0,2,2017,39938.0,12
3,0,46,0,2,2017,38774.0,12
4,1,13,0,11,2017,33769.0,12


In [13]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# ==============================
# Sort properly
# ==============================
df = df.sort_values(
    ["inst_code", "branch_code", "catgen_id", "year"]
).reset_index(drop=True)

# ==============================
# Feature Engineering
# ==============================

df["prev_cutoff"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(1)
)

df["rolling_mean_3"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .transform(lambda x: x.rolling(3).mean())
)

print("Rows before drop:", len(df))

df = df.dropna().reset_index(drop=True)

print("Rows after drop :", len(df))

# ==============================
# Define features
# ==============================
features = [
    "inst_code",
    "branch_code",
    "co_education",
    "dist_code",
    "year",
    "catgen_id",
    "prev_cutoff",
    "rolling_mean_3"
]

# ==============================
# Time-based split
# ==============================
train = df[df["year"] < 2023]
test = df[df["year"] == 2024]

X_train = train[features]
y_train = train["cutoff"]

X_test = test[features]
y_test = test["cutoff"]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# ==============================
# Train model
# ==============================
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ==============================
# 9️⃣ Save Model as Pickle
# ==============================

model_path = "C:\\college_recommendation_system\\CleanData\\RandomForest_cutoff_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully:", model_path)


# ==============================
# 🔟 Load Pickle Model
# ==============================

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)


# ==============================
# 1️⃣1️⃣ Predict
# ==============================

if len(X_test) > 0:

    y_pred = loaded_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Random Forest MAE:", mae)
    print("Random Forest R2 :", r2)

    results = test.copy()
    results["predicted_cutoff"] = y_pred
    results["error"] = results["cutoff"] - results["predicted_cutoff"]

    results.to_csv(
        "C:\\college_recommendation_system\\CleanData\\RandomForest_cutoff_model.csv",
        index=False
    )

    print("Predictions saved successfully")

else:
    print("⚠ No test data available for evaluation")

Rows before drop: 119307
Rows after drop : 72477
Train size: (43666, 8)
Test size: (14636, 8)
Model saved successfully: C:\college_recommendation_system\CleanData\RandomForest_cutoff_model.pkl
Random Forest MAE: 21978.95123095655
Random Forest R2 : 0.7089603196760446
Predictions saved successfully
